# **Pràctica 3: n-Grames Oberts i Classificador Ingenu de Bayes**

Aquest exercici explora els **n-Grames Oberts** com a **representació de paraules**, demostrant la seva robustesa davant del soroll i del desordre.

Definirem un **$n$-grama** com una configuració específica de $n$ lletres consecutives dins d'una paraula.

+ Per exemple, `at` i `ió` són alguns dels bigrames ($2$-grames) que formen part de la paraula `atenció`. Fent servir aquest concepte, podriem dir que la paraula `atenció` es pot representar pel conjunt de bigrames `{at, te, en, nc, ci, ió}`.

Un **$n$-grama obert** és una configuració que defineix un subconjunt ordenat —però no necessàriament contigu— de $n$ lletres dins d'una paraula.

+ Per exemple, si considerem la paraula `hello`, el conjunt de bigrames oberts és `{he, hl, ho, el, eo, ll, lo}`.
+ En moltes ocasions també és útil considerar el començament i el final d’una paraula com a casos especials, tenint en compte l’espai en blanc.
En aquest cas, per exemple, la paraula `hello` genera el següent conjunt ampliat de bigrames oberts:
`{ _h, he, hl, ho, el, eo, ll, lo, o_ }`.

> El cervell humà pot llegir frases amb les lletres internes desordenades sempre que: La primera i l’última lletra de cada paraula es mantinguin al lloc, la paraula tingui una longitud suficient, el context de la frase sigui clar. Això passa perquè, quan llegim, no analitzem cada lletra una per una, sinó que reconeixem les paraules com a formes globals i utilitzem el context semàntic per omplir els buits. El cervell fa una mena de “correcció automàtica” basant-se en les paraules que espera veure.

> Exemple: `Segns un etsdui de la Uinveristtat de Cmarbigde, el crevell pot lgegir paaulebs ambl les lleterres barajades i mab srrol sense mases dfiicultats.`

> Els n-grames oberts (open n-grams) donen una explicació molt natural de per què podem llegir/reconeixer paraules amb les lletres internes desordenades.

La definició dels $n$-grames oberts es basa en dos aspectes clau:
+ Ordenació: Les lletres han d’aparèixer en el mateix ordre que en la paraula original (per exemple, `NBK` és un 3-grama obert vàlid de la paraula `NOTEBOOK`, però `KBN` no ho és, perquè la $\text{K}$ apareix després de la $\text{B}$ i la $\text{N}$ a la paraula).
+ No contigüitat: Les lletres poden estar separades per qualsevol nombre d’altres lletres (això és el que fa que la representació sigui robusta davant de soroll, com ara errors tipogràfics o lletres que falten).

El nombre total de $n$-grames oberts per a una paraula de $L$ lletres ve donat pel coeficient binomial $\binom{L}{n}$.

Per a `NOTEBOOK` ($L=8$ i $n=3$), el total és:

$$\binom{8}{3} = \frac{8!}{3!(8-3)!} = \frac{8 \times 7 \times 6}{3 \times 2 \times 1} = \mathbf{56}$$.

## Part 1: Extracció de Característiques — $N$-grames oberts

### Tasca 1

Implementa una per extreure els $n$-grames oberts d’una paraula, que ens servirà com un nou tipus de representació de les paraules d'un text.

In [29]:
from itertools import combinations

def get_open_ngrams(word: str, n: int, include_boundaries: bool = True) -> set:
    """Genera un conjunt d'n-grams oberts per a una paraula donada, amb un tractament específic dels límits.

    Args:
        word (str): La paraula d'entrada.
        n (int): L'ordre de l'n-gram (per exemple, 2 per a bigrames, 3 per a trigrames).
        include_boundaries (bool): Si és True, afegeix un guió baix '_' al principi i al
                                final de la paraula per incloure els límits inicial/final,
                                però només combina el caràcter de límit amb el primer/últim
                                caràcter real de la paraula.

    Returns:
        set: Un conjunt d'n-grams oberts únics.
    """
    open_ngrams = set()
    
    if include_boundaries:
        extended_word = '_' + word + '_'
        last_idx = len(extended_word) - 1
        
        for combo in combinations(range(len(extended_word)), n):
            if 0 in combo and 1 not in combo:
                continue
            if last_idx in combo and (last_idx - 1) not in combo:
                continue
            
            ngram = ''.join(extended_word[i] for i in combo)
            open_ngrams.add(ngram)
    else:
        for combo in combinations(range(len(word)), n):
            ngram = ''.join(word[i] for i in combo)
            open_ngrams.add(ngram)

    return open_ngrams

Executa el test de la funció `get_open_ngrams` amb la paraula `hello` i comprova que funciona:

In [30]:
word_to_test = "hello"
n_gram_order_test = 2

ngrams_with_boundaries = get_open_ngrams(word_to_test, n_gram_order_test, include_boundaries=True)

assert ngrams_with_boundaries == {'_h', 'el', 'eo', 'he', 'hl', 'ho', 'll', 'lo', 'o_'}

## Part 2: Col·lisions entre paraules

Podem avaluar la capacitat per representar paraules comprovant la **taxa de col·lisió de paraules**, que es produeix quan dues paraules diferents tenen el mateix conjunt de característiques.

Per fer-ho farem servir el fitxer `dataset.csv`, que conté frases en diferents idiomes:

In [31]:
import pandas as pd
from collections import Counter
import os

dataset_file_path = 'dataset.csv'

try:
    df = pd.read_csv(dataset_file_path)

    # Extracció de textes i etiquetes
    texts = df['Text'].tolist()
    labels = df['language'].tolist()

    print("Fitxer carregat.")
    print(f"Nombre de frases: {len(texts)}")
    print(f"Exemple de frase: {texts[0]}")
    print(f"Exemple d'etiqueta: {labels[0]}")
    print("\nDistribució de Llengues:")
    print(Counter(labels))

except FileNotFoundError:
    print(f"Error: Fitxer '{dataset_file_path}' no trobat.")
    print("Assegura't que dataset.csv està accessible.")
except Exception as e:
    print(f"Error al carregar el fitxer: {e}")

Fitxer carregat.
Nombre de frases: 22000
Exemple de frase: klement gottwaldi surnukeha palsameeriti ning paigutati mausoleumi surnukeha oli aga liiga hilja ja oskamatult palsameeritud ning hakkas ilmutama lagunemise tundemärke  aastal viidi ta surnukeha mausoleumist ära ja kremeeriti zlíni linn kandis aastatel – nime gottwaldov ukrainas harkivi oblastis kandis zmiivi linn aastatel – nime gotvald
Exemple d'etiqueta: Estonian

Distribució de Llengues:
Counter({'Estonian': 1000, 'Swedish': 1000, 'Thai': 1000, 'Tamil': 1000, 'Dutch': 1000, 'Japanese': 1000, 'Turkish': 1000, 'Latin': 1000, 'Urdu': 1000, 'Indonesian': 1000, 'Portugese': 1000, 'French': 1000, 'Chinese': 1000, 'Korean': 1000, 'Hindi': 1000, 'Spanish': 1000, 'Pushto': 1000, 'Persian': 1000, 'Romanian': 1000, 'Russian': 1000, 'English': 1000, 'Arabic': 1000})


### Tasca 2

Escriu un codi en Python que:

1. Extreu totes les paraules úniques de `texts` i posa-les a un conjunt que es dirà `unique_words`
2. Genera, per a cada paraula, els bigrames oberts ($n$=2) i conta quants conjunts de bigrames únics hi ha al dataset.
3. Detecta si hi ha col·lisions al conjunt de dades, és a dir, casos en què diferents paraules tenen exactament el mateix conjunt d’$n$-grames.
4. Imprimeix:
    + El total de paraules **úniques** extretes.
    + El total de conjunts **únics** d’n-grames generats.
    + El nombre de paraules úniques implicades en col·lisions.
    + 5 exemples de parellles de paraules en col·lisió.

In [25]:
import re
from collections import defaultdict

# 1. Extreu totes les paraules úniques de 'texts'
unique_words = set()
for text in texts:
    text_lower = text.lower()
    words_in_text = re.findall(r'\b\w+\b', text_lower)
    unique_words.update(words_in_text)

# 2. Genera els bigrames oberts (n=2) amb límits per a cada paraula única
word_to_ngrams = {}
ngrams_to_words = defaultdict(list)

for word in unique_words:
    ngrams = frozenset(get_open_ngrams(word, 2, include_boundaries=True))
    word_to_ngrams[word] = ngrams
    ngrams_to_words[ngrams].append(word)

# 3. Identifica i compta les col·lisions
colliding_words = set()
collision_examples = []

for ngrams, words in ngrams_to_words.items():
    if len(words) > 1:
        colliding_words.update(words)
        for i in range(len(words)):
            for j in range(i + 1, len(words)):
                collision_examples.append((words[i], words[j]))

# 4. Compta i imprimeix: el notal de paraules úniques extretes;
# el nombre total de conjunts únics d'n-grames generats;
# el nombre total de paraules úniques implicades en col·lisions;
# 5 exemples de paraules que col·lisionen

print(f"Total de paraules úniques: {len(unique_words)}")
print(f"Total de conjunts únics d'n-grames: {len(ngrams_to_words)}")
print(f"Nombre de paraules implicades en col·lisions: {len(colliding_words)}")
print(f"\n5 exemples de parelles de paraules en col·lisió:")
for i, (word1, word2) in enumerate(collision_examples[:5]):
    print(f"  {i+1}. '{word1}' <-> '{word2}'")

Total de paraules úniques: 278630
Total de conjunts únics d'n-grames: 277959
Nombre de paraules implicades en col·lisions: 1335

5 exemples de parelles de paraules en col·lisió:
  1. 'لتعديل' <-> 'للتعديل'
  2. 'președinți' <-> 'președinții'
  3. 'جوړلو' <-> 'جوړولو'
  4. 'индии' <-> 'инди'
  5. 'antena' <-> 'antenna'


## Part 2: Classificador Ingenu de Bayes

L'objectiu és fer un classificador que sigui capaç de predir la llengua d'una frase a partir de representar la frase com el conjunt de bigrames de les seves paraules.

Per fer-ho començarem dividint el nostre dataset en una part de *training* i una de *test*.

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)

print(f"Training texts length: {len(X_train)}")
print(f"Test texts length: {len(X_test)}")
print(f"Training labels length: {len(y_train)}")
print(f"Test labels length: {len(y_test)}")

Training texts length: 17600
Test texts length: 4400
Training labels length: 17600
Test labels length: 4400


### Tasca 3: Preparació de les dades
1. Extreu els bigrames oberts de cada un dels textos del conjunt d'entrenament (els bigrames oberts d'un text és el resultat de la unió dels conjunts de bigrames de totes les seves paraules).
2. Un cop calculats, representa cada text d'entrenament amb una llista (`X_train_ngrams[:][:]`) de bigrames enlloc d'un conjunt. El primer índex és l'índex del text i el segon la llista dels seus bigrames.
3. Mostra la llista que correspon al primer text d'entrenament.

Indicació: Per extreure totes les paraules d'un text pots fer servir aquest codi:

```python
text_lower = text.lower()
words_in_text = re.findall(r'\b\w+\b', text_lower)
```


In [32]:
# Extreu els n-grames oberts dels textos del conjunt de training a la variable X_train_ngrams[:][:]
X_train_ngrams = []

for text in X_train:
    text_lower = text.lower()
    words_in_text = re.findall(r'\b\w+\b', text_lower)
    
    text_ngrams = []
    for word in words_in_text:
        word_ngrams = get_open_ngrams(word, 2, include_boundaries=True)
        text_ngrams.extend(word_ngrams)
    
    X_train_ngrams.append(text_ngrams)

print(f"Primer text d'entrenament té {len(X_train_ngrams[0])} bigrames")
print(f"Exemple dels primers 20 bigrames: {X_train_ngrams[0][:20]}")

Primer text d'entrenament té 417 bigrames
Exemple dels primers 20 bigrames: ['ส_', '_ส', 'ะส', 'ส_', 'ปร', 'มส', 'ปส', 'รส', 'มะ', '_ม', 'ปะ', 'มร', 'ระ', 'มป', 'ทธ', '_ท', 'ธ_', '_ฮ', 'ฮล', 'ฮอ']


Extreu els n-grames oberts dels textos del conjunt de test (`X_test_ngrams`[:][:]).


In [33]:
# Extreu els n-grames oberts dels textos del conjunt de test a la variable X_test_ngrams[:][:]
X_test_ngrams = []

for text in X_test:
    text_lower = text.lower()
    words_in_text = re.findall(r'\b\w+\b', text_lower)
    
    text_ngrams = []
    for word in words_in_text:
        word_ngrams = get_open_ngrams(word, 2, include_boundaries=True)
        text_ngrams.extend(word_ngrams)
    
    X_test_ngrams.append(text_ngrams)

print(f"Nombre de textos de test processats: {len(X_test_ngrams)}")

Nombre de textos de test processats: 4400


El següent objectiu és aplicar un **classificador ingenu de Bayes** per detectar les llegües dels textos del conjunt de test.

Per fer-ho hem de convertir els $n$-grames oberts extrets en **representacions numèriques** i ho farem amb el mètode TF-IDF, preparant-los per al classificador.

> El **TF-IDF** (de Term Frequency – Inverse Document Frequency) és una tècnica molt utilitzada en processament del llenguatge natural per representar textos de manera numèrica i mesurar la importància de cada element del text (en el nostre cas bigrames) dins d’un conjunt de documents.

TF-IDF combina dues idees simples:

+ TF — Term Frequency (freqüència del terme):  Mesura quantes vegades apareix un element o terme dins d’un document.

$$TF(t, d) = \frac{\text{nombre de vegades que el terme } t \text{ apareix a } d}{\text{nombre total de paraules al document } d}$$

+ IDF — Inverse Document Frequency (freqüència inversa del document): Mesura com d’especial és un element dins del conjunt total de documents.

$$ IDF(t) = \log\left(\frac{N}{1 + n_t}\right) $$

On:
+ $N$ = nombre total de documents
+ $n_t$ = nombre de documents on apareix l'element $t$

👉 Si una paraula (o bigrama) apareix a gairebé tots els documents (com `el`, `una`, `de`), el seu IDF és baix.

👉 Si només apareix en pocs, el seu IDF és alt — i, per tant, és més discriminativa.

El **TF-IDF** és:

$$TF\text{-}IDF(t, d) = TF(t, d) \times IDF(t)$$

Així, el elements:
+ freqüents dins d’un text (alt TF)
+ però poc freqüents en la resta del corpus (alt IDF)

reben més pes en la representació numèrica.

Suposa dos textos:
1. “El gat dorm al sofà.”
2. “El gos juga al parc.”

Les paraules `el` i `al` apareixen a tots dos → TF alt però IDF baix.
Les paraules `gat`, `gos`, `sofà` o `parc` apareixen només a un → TF moderat però IDF alt → més importants per diferenciar els textos.



In [34]:
# TF-IDF ja està inclòs a `scikit-learn`, una de les llibreries més
# populars per a machine learning i el podem fer servir així en el
# nostre cas:

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(analyzer=lambda x: x)

vectorizer.fit(X_train_ngrams)

X_train_tfidf = vectorizer.transform(X_train_ngrams)
X_test_tfidf = vectorizer.transform(X_test_ngrams)

print("TF-IDF vectorization complete.")
print(f"Shape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape} \n")

TF-IDF vectorization complete.
Shape of X_train_tfidf: (17600, 807585)
Shape of X_test_tfidf: (4400, 807585) 



Les matrius que ha generat la cel·la anterior estan guardats amb una estructura de python que es diu "Compressed Sparse Row sparse matrix".

Pensa que tens una taula enorme de nombres, però la majoria són zeros. Guardar tots aquests zeros és una pèrdua de memòria i de temps.

Per això en Python (amb scipy) sovint es fan servir matrius disperses (sparse matrices), i una de les més habituals és el format Compressed Sparse Row (CSR).

En lloc de guardar tots els elements de la matriu, una CSR només guarda:
+ Els valors no zero
+ La columna de cada valor no zero
+ On comença i acaba cada fila dins d’aquestes llistes

Però tu, com a usuari, no cal que et preocupis gaire de com ho fa per dins:
la tractes gairebé com si fos una matriu de NumPy, però amb algunes diferències.

### Tasca 4: Implementació d'un classificador ingenu de Bayes

Has d'entrenar un classificador Naive Bayes amb les característiques TF-IDF del conjunt d'entrenament i les seves etiquetes de llengua corresponents.

El model calcula la probabilitat que un text $d$ pertanyi a una llengua $c$:

$$P(c \mid d) \propto P(c) \prod_{i=1}^{V} P(w_i \mid c)^{\, f_i}$$

On:
+ $P(c)$: probabilitat prèvia de la classe (per exemple, % de textos d'una llengua; en el nostre cas totes les llengües tenen la mateixa probabilitat).
+ $w_i$: bigrama i-èssim del vocabulari.
+ $f_i$: nombre de vegades que el bigrama $w_i$ apareix al text $d$.
+ $P(w_i \mid c)$: probabilitat que el bigrama $w_i$ aparegui en textos de la classe $c$.
+ $V$: mida del vocabulari.

El classificador escull la classe amb probabilitat més alta.

#### Com es calcula $P(w_i \mid c)$?

Normalment es calcula amb un model multinomial amb Laplace *smoothing* (per evitar zeros):

$$P(w_i \mid c) =
\frac{N_{i,c} + 1}{\sum_{j=1}^{V} N_{j,c} + V}$$

On:
+ $N_{i,c}$: nombre total de vegades que la paraula $w_i$ apareix en tots els documents de la classe c.
+ $V$: nombre total de paraules diferents del vocabulari.

Aquesta fórmula ve directament de la distribució multinomial, perquè compta freqüències de paraules dins d’una “bossa” pròpia de cada classe.

Fes una implementació teva del model multinomial, és a dir, de les funcions que creen el model i apliquen el model al test:

 `MultinomialNBfit(X_train_tfidf, y_train))`

 i

 `MultinomialNBpredict(model, X_test_tfidf, y_test))`

Imprimeix quina `accuracy` obtens. Si tot funciona correctament, hauries d'aconseguir una *accuracy* per sobre el 90%.

In [36]:
import numpy as np
from sklearn.metrics import accuracy_score

def MultinomialNBfit(X, y):
    """
    Entrena el classificador Naive Bayes.

    Args:
        X (sparse matrix): Matriu de característiques (e.g., TF-IDF) d'entrenament.
        y (array-like): Etiquetes de classe per a cada mostra.
    """
    classes = np.unique(y)
    n_classes = len(classes)
    n_features = X.shape[1]
    
    class_log_prior = np.zeros(n_classes)
    feature_log_prob = np.zeros((n_classes, n_features))
    
    for idx, cls in enumerate(classes):
        mask = np.array([label == cls for label in y])
        X_cls = X[mask]
        
        class_log_prior[idx] = np.log(X_cls.shape[0] / X.shape[0])
        
        feature_count = np.asarray(X_cls.sum(axis=0)).ravel()
        
        smoothed_count = feature_count + 1
        smoothed_total = smoothed_count.sum()
        
        feature_log_prob[idx, :] = np.log(smoothed_count / smoothed_total)
    
    model = {
        'classes': classes,
        'class_log_prior': class_log_prior,
        'feature_log_prob': feature_log_prob
    }
    
    return model

def MultinomialNBpredict(model, X, y):
    """
    Aplica el classificador Naive Bayes.

    Args:
        X (sparse matrix): Matriu de característiques (e.g., TF-IDF) de test.
        y (array-like): Etiquetes de classe per a cada mostra del test.
    """
    classes = model['classes']
    class_log_prior = model['class_log_prior']
    feature_log_prob = model['feature_log_prob']
    
    log_prob = X.dot(feature_log_prob.T) + class_log_prior
    
    y_pred = classes[np.argmax(log_prob, axis=1)]
    
    accuracy = accuracy_score(y, y_pred)
    
    return accuracy

model = MultinomialNBfit(X_train_tfidf, y_train)
accuracy = MultinomialNBpredict(model, X_test_tfidf, y_test)

print(f"Accuracy del classificador: {accuracy:.4f} ({accuracy*100:.2f}%)")

Accuracy del classificador: 0.9666 (96.66%)
